# Computer Vision Lab

## Tracking — Where did the object move?

**Mission**

Tracking connects detections across video frames and gives them temporary IDs.

In [ ]:
# Install Ultralytics because it supplies object detection and the ByteTrack tracker.
!pip -q install ultralytics


In [ ]:
# urlretrieve downloads the public source video.
from urllib.request import urlretrieve
# run executes ffmpeg so browser-compatible videos can be created.
from subprocess import run
# cv2 reads frames and writes the intermediate annotated video.
import cv2
# Video and display render H.264 videos inside Colab.
from IPython.display import Video, display
# YOLO loads the pretrained object detector.
from ultralytics import YOLO


### Predict

When two people overlap, will the tracker keep the same ID for each person?

In [ ]:
# VIDEO_URL is the public pedestrian-video source.
VIDEO_URL = "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/vtest.avi"
# VIDEO_PATH is the downloaded local source filename.
VIDEO_PATH = "vtest.avi"
# OUTPUT_PATH is the first annotated tracking video, before web conversion.
OUTPUT_PATH = "tracked_people.mp4"
# Download the source video once for the following preview and tracking steps.
urlretrieve(VIDEO_URL, VIDEO_PATH)


### Watch the original video

This is the unannotated source. Watch it before running the tracker.


In [ ]:
# SOURCE_PLAYABLE_PATH is an H.264 copy that browsers can play reliably.
SOURCE_PLAYABLE_PATH = "original_video_playable.mp4"
# ffmpeg converts the original video: y overwrites old files, c:v chooses H.264, pix_fmt improves browser compatibility.
run(["ffmpeg", "-y", "-loglevel", "error", "-i", VIDEO_PATH, "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", SOURCE_PLAYABLE_PATH], check=True)
# Show the unannotated source video before tracking.
display(Video(SOURCE_PLAYABLE_PATH, embed=True, width=900))


### Set the experiment

Use fewer frames for a faster first run. Adjust confidence only after watching the first result.

In [ ]:
# CONFIDENCE is the minimum detection score used before an object can be tracked.
CONFIDENCE = 0.35
# MAX_FRAMES limits processing time; lower it for a faster test.
MAX_FRAMES = 180
# Load YOLO11's lightweight object-detection weights.
model = YOLO("yolo11n.pt")


### Run tracking

This cell creates a new video with boxes and temporary tracking IDs.

In [ ]:
# RAW_OUTPUT_PATH uses an AVI container because OpenCV writes MJPEG there reliably.
RAW_OUTPUT_PATH = "tracked_people.avi"
# model.track(...) detects objects, assigns temporary IDs, and yields one result per frame.
# persist=True keeps tracker state from one frame to the next.
# stream=True avoids storing every video result in memory at once.
stream = model.track(source=VIDEO_PATH, tracker="bytetrack.yaml", persist=True, conf=CONFIDENCE, stream=True, verbose=False)
# writer starts empty because output width and height are learned from the first annotated frame.
writer = None
# enumerate supplies both the frame number and the model result for that frame.
for frame_number, result in enumerate(stream):
    # Stop once the chosen frame limit is reached.
    if frame_number >= MAX_FRAMES:
        break
    # result.plot() draws boxes, class labels, confidence, and tracking IDs on this frame.
    annotated = result.plot()
    # Create the output writer exactly once, using the first frame's dimensions.
    if writer is None:
        height, width = annotated.shape[:2]
        # MJPG is a broadly supported intermediate codec for OpenCV video writing.
        writer = cv2.VideoWriter(RAW_OUTPUT_PATH, cv2.VideoWriter_fourcc(*"MJPG"), 25, (width, height))
        # Stop clearly if the runtime cannot create the output video.
        if not writer.isOpened():
            raise RuntimeError("OpenCV could not create the intermediate tracking video.")
    # Add the annotated frame to the intermediate video.
    writer.write(annotated)
# Finish the file so ffmpeg and the browser can read it.
if writer is not None:
    writer.release()
    # frame_number starts at zero, so add one when reporting how many frames were written.
    print(f"Saved {RAW_OUTPUT_PATH} after {frame_number + 1} frames.")
else:
    raise RuntimeError("No frames were processed. Check the video URL and model download.")


### Inspect the evidence

Watch for an ID switch after an object leaves the frame or overlaps another object.

### Make the result playable

The tracking video is converted to H.264 so it plays inside Colab.


In [ ]:
# PLAYABLE_PATH is the final H.264 tracking video that Colab can play.
PLAYABLE_PATH = "tracked_people_playable.mp4"
# Convert the OpenCV video to H.264 with browser-safe pixels and fast-start metadata.
run(["ffmpeg", "-y", "-loglevel", "error", "-i", RAW_OUTPUT_PATH, "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart", PLAYABLE_PATH], check=True)


In [ ]:
# Show the final video with object boxes and temporary tracking IDs.
display(Video(PLAYABLE_PATH, embed=True, width=900))


### Change one thing

Try `MAX_FRAMES = 60`, then change `CONFIDENCE`. What changes in the IDs and boxes?

### Evidence

When would anonymous tracking be enough, without identifying a person?